In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import dice_ml
from dice_ml import Dice
import warnings
from tqdm import tqdm

warnings.filterwarnings('ignore')

print("=" * 80)
print("Step 3: DiCE Counterfactual Explanation Generation (Genetic Method)")
print("=" * 80)

# ============================================================================
# Project paths
# ============================================================================
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
MODEL_DIR = RESULTS_DIR / 'model'
TABLE_DIR = RESULTS_DIR / 'table'

print("Project paths ready")

# ============================================================================
# 1. Data and Model Loading
# ============================================================================
print("\n[Step 1] Loading Data and Model")

# 1-1. Load the ratio-based, cleaned modeling dataset (has ID + PERF_12M + ratio features)
df_ratio = pd.read_csv(DATA_DIR / 'selected_data_for_modeling_full_ratio_clean.csv')

# 1-2. Load the ORIGINAL raw dataset too — needed to recover SIC_CD_3 (industry
#      code), which was dropped during the ratio-conversion step in Step1.
df_raw = pd.read_csv(DATA_DIR / '202207_corpor_CB.csv', usecols=['ID', 'SIC_CD_3'])

# 1-3. Merge industry code back in, keyed on ID
df_merged = df_ratio.merge(df_raw, on='ID', how='left')

print(f"Ratio dataset shape: {df_ratio.shape}")
print(f"Merged with industry codes: {df_merged.shape}")
print(f"Rows with missing SIC_CD_3 after merge: {df_merged['SIC_CD_3'].isnull().sum()}")

# 1-4. Load model, features, threshold from Step2
model = joblib.load(MODEL_DIR / 'base_model_final_full.pkl')
feature_names = joblib.load(MODEL_DIR / 'selected_features_final_full.pkl')
threshold = joblib.load(MODEL_DIR / 'base_model_threshold_final_full.pkl')

print(f"\nModel loaded: base_model_final_full.pkl")
print(f"Decision threshold: {threshold:.4f}")
print(f"Number of features: {len(feature_names)}")

# ============================================================================
# 2. Define Analysis Targets — bankrupt firms, restricted to the 4 selected
#    industries (G46, G47, L68, F42), matching the industry-selection logic
#    established in Step1's original exploration.
# ============================================================================
print("\n[Step 2] Defining Analysis Targets")

TARGET_SIC_CODES = ['G46', 'G47', 'L68', 'F42']

target_df = df_merged[
    (df_merged['PERF_12M'] == 1) &
    (df_merged['SIC_CD_3'].isin(TARGET_SIC_CODES))
].copy()

X_target = target_df[feature_names]
target_ids = target_df['ID'].values
target_sics = target_df['SIC_CD_3'].values

print(f"Total bankrupt firms in full corpus: {(df_merged['PERF_12M']==1).sum()}")
print(f"Bankrupt firms restricted to {TARGET_SIC_CODES}: {len(X_target)}")

print("\nBreakdown by industry:")
print(target_df['SIC_CD_3'].value_counts())

# ============================================================================
# 3. DiCE Object Initialisation (Genetic Algorithm)
# ============================================================================
print("\n[Step 3] Initialising DiCE Object (Genetic Algorithm)")

d = dice_ml.Data(
    dataframe=df_ratio.drop('ID', axis=1),
    continuous_features=feature_names,
    outcome_name='PERF_12M'
)

m = dice_ml.Model(model=model, backend='sklearn', model_type='classifier')

exp = Dice(d, m, method='genetic')

print("Configuration complete: method='genetic'")

# ============================================================================
# 4. Immutable Features — updated for ratio-transformed column names.
#
#    The paper's rationale (Section 3.4.2): prior-year figures cannot be
#    retroactively altered. After Step1's ratio conversion:
#      - Prior-period LEVEL variables (e.g. FN1_13_1, FN2_1_1) no longer
#        exist as standalone features — they were consumed as the denominator
#        in year-over-year growth rates (Group D in Step1).
#      - Those growth-rate features (e.g. 'revenue_growth_rate') are treated
#        as immutable in spirit: the prior-year base that anchors them is
#        fixed, so we do not let DiCE freely invent a growth-rate value
#        disconnected from a real current-period balance-sheet change.
#      - The CURRENT-period components that feed other ratios (e.g. the
#        numerator in 'FN1_11_to_assets') remain mutable, since a firm CAN
#        act on its present-period receivables, inventory, etc.
# ============================================================================
IMMUTABLE_FEATURES = [
    'asset_growth_rate',
    'revenue_growth_rate',
    'operating_income_growth',
    'net_income_growth',
    'equity_growth_rate',
]

# Sanity check: make sure every listed immutable feature actually exists in
# the trained feature set (guards against silent no-ops if names drift again)
missing_immutables = [f for f in IMMUTABLE_FEATURES if f not in feature_names]
if missing_immutables:
    raise ValueError(f"Immutable features not found in feature_names: {missing_immutables}. "
                      f"Check Step1's ratio-conversion column names.")

features_to_vary = [f for f in feature_names if f not in IMMUTABLE_FEATURES]

print(f"\nImmutable features ({len(IMMUTABLE_FEATURES)}): {IMMUTABLE_FEATURES}")
print(f"Mutable features ({len(features_to_vary)}): first 5 = {features_to_vary[:5]}")

# ============================================================================
# 5. Counterfactual (CF) Generation Loop
# ============================================================================
print("\n[Step 4] Starting Counterfactual Generation")
print(f"Processing {len(X_target)} firms (4-industry subset)")

cf_results_list = []
cf_failed_ids = []

for i in tqdm(range(len(X_target)), desc="Processing"):
    try:
        row = X_target.iloc[i]
        current_id = target_ids[i]
        current_sic = target_sics[i]

        query_instance = row.to_frame().T

        original_prob = model.predict_proba(query_instance)[0, 1]

        dice_exp = exp.generate_counterfactuals(
            query_instance,
            total_CFs=4,
            desired_class=0,
            features_to_vary=features_to_vary,
            verbose=False,
        )

        cf_df = dice_exp.cf_examples_list[0].final_cfs_df

        if cf_df is not None and not cf_df.empty:
            for cf_idx, cf_row in cf_df.iterrows():
                cf_features = cf_row[feature_names].values.reshape(1, -1)
                cf_prob = model.predict_proba(cf_features)[0, 1]

                result_entry = {
                    'ID': current_id,
                    'SIC_CD_3': current_sic,
                    'CF_Number': cf_idx + 1,
                    'Original_Proba': original_prob,
                    'Target_Proba': cf_prob
                }

                for feat in feature_names:
                    orig_val = row[feat]
                    cf_val = cf_row[feat]
                    result_entry[f'Original_{feat}'] = orig_val
                    result_entry[f'CF_{feat}'] = cf_val
                    result_entry[f'Change_{feat}'] = cf_val - orig_val

                cf_results_list.append(result_entry)
        else:
            cf_failed_ids.append({'ID': current_id, 'SIC_CD_3': current_sic})

    except Exception as e:
        cf_failed_ids.append({'ID': current_id, 'SIC_CD_3': current_sic, 'error': str(e)})
        continue

# ============================================================================
# 6. Save Results
# ============================================================================
print("\n" + "=" * 80)
print("[Step 5] Saving Results")
print("-" * 80)

final_cf_df = pd.DataFrame(cf_results_list)

total_target = len(X_target)
success_cnt = final_cf_df['ID'].nunique() if not final_cf_df.empty else 0
fail_cnt = len(cf_failed_ids)

print(f"Total analysis targets: {total_target}")
print(f"Successful firms: {success_cnt} (success rate: {success_cnt/total_target*100:.1f}%)")
print(f"Failed firms: {fail_cnt}")

if not final_cf_df.empty:
    output_path = DATA_DIR / 'cf_results_4industry.csv'
    final_cf_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\nResults saved: {output_path.relative_to(PROJECT_ROOT)}")

    print("\n[Success rate by industry]")
    success_by_sic = final_cf_df.groupby('SIC_CD_3')['ID'].nunique()
    total_by_sic = target_df.groupby('SIC_CD_3')['ID'].nunique()
    rate_by_sic = (success_by_sic / total_by_sic * 100).round(1)
    print(pd.DataFrame({'success': success_by_sic, 'total': total_by_sic, 'rate_%': rate_by_sic}))

    print("\n[Sample Output (First Successful Firm)]")
    sample = final_cf_df.iloc[0]
    print(f"ID: {sample['ID']} ({sample['SIC_CD_3']})")
    print(f"Bankruptcy probability change: {sample['Original_Proba']:.4f} -> {sample['Target_Proba']:.4f}")
else:
    print("\n[Warning] No CFs generated. Please check the model and data.")

if cf_failed_ids:
    failed_path = TABLE_DIR / 'cf_failed_ids_4industry.csv'
    pd.DataFrame(cf_failed_ids).to_csv(failed_path, index=False)
    print(f"Failed ID list saved: {failed_path.relative_to(PROJECT_ROOT)}")

print("\nStep 3 complete. Proceed to Step 4 (CF quality evaluation / selection).")

Step 3: DiCE Counterfactual Explanation Generation (Genetic Method)
Project paths ready

[Step 1] Loading Data and Model
Ratio dataset shape: (147724, 64)
Merged with industry codes: (147724, 65)
Rows with missing SIC_CD_3 after merge: 21

Model loaded: base_model_final_full.pkl
Decision threshold: 0.4683
Number of features: 62

[Step 2] Defining Analysis Targets
Total bankrupt firms in full corpus: 2070
Bankrupt firms restricted to ['G46', 'G47', 'L68', 'F42']: 829

Breakdown by industry:
SIC_CD_3
G46    308
L68    199
G47    197
F42    125
Name: count, dtype: int64

[Step 3] Initialising DiCE Object (Genetic Algorithm)
Configuration complete: method='genetic'

Immutable features (5): ['asset_growth_rate', 'revenue_growth_rate', 'operating_income_growth', 'net_income_growth', 'equity_growth_rate']
Mutable features (57): first 5 = ['FN3_3', 'FN3_6', 'FN3_10', 'FN1_1_to_assets', 'FN1_2_to_assets']

[Step 4] Starting Counterfactual Generation
Processing 829 firms (4-industry subset)


Processing: 100%|██████████████████████████████████████████████████████████████████| 829/829 [1:18:34<00:00,  5.69s/it]



[Step 5] Saving Results
--------------------------------------------------------------------------------
Total analysis targets: 829
Successful firms: 640 (success rate: 77.2%)
Failed firms: 189

Results saved: data\cf_results_4industry.csv

[Success rate by industry]
          success  total  rate_%
SIC_CD_3                        
F42           105    125    84.0
G46           247    308    80.2
G47           152    197    77.2
L68           136    199    68.3

[Sample Output (First Successful Firm)]
ID: 126732 (L68)
Bankruptcy probability change: 0.9903 -> 0.5243
Failed ID list saved: results\table\cf_failed_ids_4industry.csv

Step 3 complete. Proceed to Step 4 (CF quality evaluation / selection).
